
# Vet filtered Lantern candidates with DP2 deep-coadd RGB cutouts

This notebook follows the strategy of `dp2_deep_coadd_RGB_cutout.ipynb`, adapted to the
filtered Lantern candidate list.

It is designed to run on the **RSP / data.lsst.cloud** with the LSST kernel.

For each candidate, it:

- reads the filtered candidate list;
- fetches DP2 **deep coadd** cutouts at the candidate position;
- makes two color images:
  - **IRG** = `(i, r, g)`
  - **ZIR** = `(z, i, r)`
- displays candidates one by one;
- annotates each panel with:
  - `locus_id`
  - rank / score
  - RA, Dec
  - mean science-image magnitude (`i_mag` from `lsst_diaObject_i_scienceFluxMean`)
  - Gaia / MILLiquas flags.

This notebook **does not save cutouts**. It only fetches and displays them.


In [ ]:

import io
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy import units as u
from astropy.visualization import AsinhStretch, ImageNormalize
from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth
from pyvo.dal.adhoc import SodaQuery

%matplotlib inline


In [ ]:

# -------------------- INPUTS --------------------

# Use the filtered list from the previous candidate-analysis notebook.
CANDIDATES_FILE = "candidate_analysis/second_pass_candidates_filtered.parquet"

# Deep-coadd cutout radius in degrees.
# 0.002 deg = 7.2 arcsec.
CUTOUT_RADIUS = 0.002

# Which candidates to display.
START = 0
N_SHOW = 20

# Plot controls
FIGSIZE = (10, 4)
ASINH_A = 0.002
VMIN = -0.03
VMAX = 3000

# If True, continue even when one band is missing.
SKIP_MISSING = True


In [ ]:

# Load candidate list

cand = pd.read_parquet(CANDIDATES_FILE).copy()

required = {"locus_id", "ra", "dec", "second_pass_score"}
missing = required - set(cand.columns)
if missing:
    raise KeyError(f"Missing required candidate columns: {sorted(missing)}")

cand = cand.sort_values("second_pass_score", ascending=False).reset_index(drop=True)
cand["rank"] = np.arange(1, len(cand) + 1)

print(f"Candidates loaded: {len(cand):,}")
print("Columns:")
print(list(cand.columns))


In [ ]:

# Initialize RSP discovery client

discovery = RSPDiscovery("dp2")
sia_client = discovery.get_sia_client()


In [ ]:

# Helper functions adapted from dp2_deep_coadd_RGB_cutout.ipynb

def get_dl_result(band, ra, dec, radius=CUTOUT_RADIUS):
    circle = (ra, dec, radius)
    results = sia_client.search(
        pos=circle,
        calib_level=3,
        dpsubtype='lsst.deep_coadd',
    )

    band_arr = results['lsst_band']

    try:
        index = int(np.where(band_arr == band)[-1][-1])
    except Exception:
        print(f'Band {band} does not exist at RA={ra}, Dec={dec}.')
        return None

    dl_result = discovery.get_datalink_results(results[index])
    return dl_result


def get_cutout(band, ra, dec, radius=CUTOUT_RADIUS):
    dl_result = get_dl_result(band, ra, dec, radius)
    if dl_result is None:
        return None

    sq = SodaQuery.from_resource(
        dl_result,
        dl_result.get_adhocservice_by_id("cutout-sync"),
        session=get_pyvo_auth(),
    )
    sq.circle = (ra * u.deg, dec * u.deg, radius * u.deg)

    cutout_bytes = sq.execute_stream().read()
    sq.raise_if_error()

    cutout = read_archive(io.BytesIO(cutout_bytes))
    return cutout


def normalize_band(image, asinh_a=ASINH_A, vmin=VMIN, vmax=VMAX):
    data = image.array
    norm = ImageNormalize(
        vmin=vmin,
        vmax=vmax,
        stretch=AsinhStretch(a=asinh_a),
        clip=True,
    )
    return norm(data)


def combine_rgb(R_image, G_image, B_image):
    R = normalize_band(R_image)
    G = normalize_band(G_image)
    B = normalize_band(B_image)
    return np.dstack([R, G, B])


def fetch_candidate_cutouts(ra, dec, bands=("g", "r", "i", "z"), radius=CUTOUT_RADIUS):
    cutouts = {}
    for band in bands:
        try:
            cutouts[band] = get_cutout(band, ra, dec, radius=radius)
        except Exception as e:
            print(f"Failed to fetch {band}-band cutout: {e}")
            cutouts[band] = None
    return cutouts


In [ ]:

# Text helpers

def fmt_value(x, fmt=".3f"):
    if x is None or pd.isna(x):
        return "nan"
    return format(x, fmt)


def yesno(x):
    if pd.isna(x):
        return "nan"
    return "Y" if bool(x) else "N"


def candidate_header(row):
    parts = [
        f"rank={int(row['rank'])}",
        f"locus_id={row['locus_id']}",
        f"score={fmt_value(row.get('second_pass_score'), '.3f')}",
    ]
    return " | ".join(parts)


def candidate_subtext(row):
    lines = []
    lines.append(
        f"RA={fmt_value(row.get('ra'), '.6f')}  Dec={fmt_value(row.get('dec'), '.6f')}"
    )

    if 'i_mag' in row.index:
        lines.append(f"i_mag={fmt_value(row.get('i_mag'), '.2f')}")
    elif 'median_science_mag' in row.index:
        lines.append(f"science_mag={fmt_value(row.get('median_science_mag'), '.2f')}")

    extra = []
    if 'n_alerts' in row.index:
        extra.append(f"n_alerts={int(row['n_alerts'])}" if pd.notna(row['n_alerts']) else "n_alerts=nan")
    if 'n_bands' in row.index:
        extra.append(f"n_bands={int(row['n_bands'])}" if pd.notna(row['n_bands']) else "n_bands=nan")
    if extra:
        lines.append("  ".join(extra))

    flags = []
    if 'milliquas_match' in row.index:
        flags.append(f"MILLIQUAS={yesno(row['milliquas_match'])}")
    if 'gaia_match' in row.index:
        flags.append(f"Gaia={yesno(row['gaia_match'])}")
    if 'gaia_in_qso_candidates_any' in row.index:
        flags.append(f"Gaia_QSO={yesno(row['gaia_in_qso_candidates_any'])}")
    if 'high_pm_flag' in row.index:
        flags.append(f"high_pm={yesno(row['high_pm_flag'])}")
    if 'high_parallax_flag' in row.index:
        flags.append(f"high_plx={yesno(row['high_parallax_flag'])}")
    if flags:
        lines.append("  ".join(flags))

    return "
".join(lines)


In [ ]:

# Plot one candidate: IRG and ZIR

def show_candidate(row, radius=CUTOUT_RADIUS, figsize=FIGSIZE, skip_missing=SKIP_MISSING):
    ra = float(row['ra'])
    dec = float(row['dec'])

    cut = fetch_candidate_cutouts(ra, dec, bands=("g", "r", "i", "z"), radius=radius)

    # Needed combinations:
    # IRG -> (i, r, g)
    # ZIR -> (z, i, r)
    need_irg = [cut.get('i'), cut.get('r'), cut.get('g')]
    need_zir = [cut.get('z'), cut.get('i'), cut.get('r')]

    if any(x is None for x in need_irg + need_zir):
        msg = f"Missing one or more bands for {row['locus_id']}"
        if skip_missing:
            print(msg)
        else:
            raise RuntimeError(msg)

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # IRG
    if all(x is not None for x in need_irg):
        rgb_irg = combine_rgb(cut['i'], cut['r'], cut['g'])
        axes[0].imshow(rgb_irg, origin='lower')
        axes[0].set_title('IRG')
    else:
        axes[0].text(0.5, 0.5, 'IRG unavailable', ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_title('IRG')

    # ZIR
    if all(x is not None for x in need_zir):
        rgb_zir = combine_rgb(cut['z'], cut['i'], cut['r'])
        axes[1].imshow(rgb_zir, origin='lower')
        axes[1].set_title('ZIR')
    else:
        axes[1].text(0.5, 0.5, 'ZIR unavailable', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('ZIR')

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(candidate_header(row), y=1.02, fontsize=12)
    fig.text(0.02, -0.02, candidate_subtext(row), fontsize=10, va='top')
    plt.tight_layout()
    plt.show()

    return cut


## Preview the subset to be displayed

In [ ]:

subset = cand.iloc[START:START + N_SHOW].copy()

show_cols = [
    c for c in [
        'rank', 'locus_id', 'second_pass_score',
        'ra', 'dec', 'i_mag', 'median_science_mag',
        'milliquas_match', 'gaia_match', 'gaia_in_qso_candidates_any',
        'n_alerts', 'n_bands'
    ] if c in subset.columns
]

print(f"Displaying rows {START} to {START + len(subset) - 1} (N={len(subset)})")
subset[show_cols]


## Display candidates one by one

In [ ]:

for _, row in subset.iterrows():
    show_candidate(row)



## Optional: inspect a single candidate by rank or by `locus_id`


In [ ]:

# Example: inspect one row by table index
idx = 0
show_candidate(cand.iloc[idx])


In [ ]:

# Example: inspect one specific locus_id
# target_id = "ANT..."
# row = cand.loc[cand['locus_id'] == target_id].iloc[0]
# show_candidate(row)



## Notes

- This notebook follows the original DP2 cutout notebook's flow:
  SIA search → datalink → `cutout-sync` → RGB display.
- It fetches only the bands needed for **IRG** and **ZIR** to save time.
- It does **not** save cutout files.
- If you later decide to vet all candidates, increase `N_SHOW` or loop over the full table.
- If nearby-locus merging becomes necessary, the candidate table should be rebuilt first,
  because this notebook only visualizes the current filtered list.
